## China Job Volume

Three views: tracked-role headcount, the sector mix *among tracked roles*, and the **economy-wide** sector mix (~740M workers, NBS 2023) showing software/IT against China's large manufacturing, trade, and agriculture workforces.

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import plotly.express as px
from sector_data import get_sector_employment, SOFTWARE_SECTOR

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}
SECTOR_LABELS = {
    'tech': 'Tech / Software', 'legal': 'Legal', 'healthcare': 'Healthcare',
    'finance': 'Finance', 'engineering': 'Engineering', 'blue_collar': 'Construction',
    'agriculture': 'Agriculture', 'manufacturing': 'Manufacturing', 'services': 'Retail / Services',
}
SECTOR_COLORS = {
    'Tech / Software': '#2171b5', 'Legal': '#6a51a3', 'Healthcare': '#e6550d',
    'Finance': '#31a354', 'Engineering': '#74c476', 'Construction': '#969696',
    'Agriculture': '#8c6d31', 'Manufacturing': '#bcbddc', 'Retail / Services': '#fdae6b',
}
ECON_SECTOR_COLORS = {
    'Agriculture': '#8c6d31', 'Manufacturing': '#bcbddc', 'Construction': '#969696',
    'Information & Software/IT': '#2171b5', 'Finance & Insurance': '#31a354',
    'Healthcare & Social': '#e6550d', 'Professional & Legal Services': '#6a51a3',
    'Trade (Retail & Wholesale)': '#fdae6b', 'Education': '#f768a1',
    'Public Administration': '#74c476', 'Other Services': '#d9d9d9',
}

df = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_china_data.csv')
vol = df[df['career_stage'] == 'mid'][['role', 'sector', 'employed_thousands']].copy()
vol['role_label'] = vol['role'].map(ROLE_LABELS)
vol['sector_label'] = vol['sector'].map(SECTOR_LABELS)

In [2]:
vol_sorted = vol.sort_values('employed_thousands', ascending=True)
fig = px.bar(
    vol_sorted, x='employed_thousands', y='role_label', orientation='h',
    color='employed_thousands', color_continuous_scale='Oranges',
    title='China: Total Employed by Tracked Role (NBS/MIIT 2023, thousands)',
    labels={'employed_thousands': 'Employed (thousands)', 'role_label': 'Role'},
)
fig.show()

In [3]:
sector_vol = vol.groupby('sector_label', as_index=False)['employed_thousands'].sum()
tech_pct = sector_vol.loc[sector_vol['sector_label'] == 'Tech / Software', 'employed_thousands'].values[0] / sector_vol['employed_thousands'].sum() * 100

fig2 = px.pie(
    sector_vol, values='employed_thousands', names='sector_label',
    color='sector_label', color_discrete_map=SECTOR_COLORS,
    title=f'China: Sector Mix Among the 10 Tracked Roles (2023)<br><sup>Tech / Software = {tech_pct:.1f}% of tracked workers</sup>',
)
fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.show()

In [4]:
econ = get_sector_employment('China')
sw_pct = econ.loc[econ['sector'] == SOFTWARE_SECTOR, 'pct_of_total'].values[0]

fig3 = px.pie(
    econ, values='employed_millions', names='sector',
    color='sector', color_discrete_map=ECON_SECTOR_COLORS,
    title=f'China: Economy-Wide Employment by Sector (~740M workers, NBS 2023)<br><sup>Information & Software/IT = {sw_pct:.1f}% of ALL Chinese jobs</sup>',
)
fig3.update_traces(textposition='inside', textinfo='percent+label')
fig3.show()